# DeepEval Demo — Evaluating LLM Outputs

Uses **DeepEval** with an Azure OpenAI judge model, in two parts:

1. **Part 1 — Excel Data Evaluation:** score pre-collected outputs read from two spreadsheets
   (**EasyTravel** transcript summaries and **PolicyBot** policy Q&A).
2. **Part 2 — Actual Data Evaluation:** paste `LLMTestCase(...)` objects produced by your live
   app and score them with the same metrics.

Run **Setup** first, then either part. The two parts use different metrics because the two
datasets have different fields.

## Setup

In [1]:
%pip install -q -U deepeval pandas openpyxl tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.6/394.6 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2

### Azure OpenAI configuration (the judge model)
## Reduce this

In [5]:
import os

# Disables DeepEval's telemetry, preventing it from sending anonymous usage and diagnostic data to the DeepEval developers.
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

# Fill in YOUR Azure OpenAI values.
os.environ["AZURE_API_KEY"] = "5xD6IeVNWL7eYqgbuxRKVATJK38hdacwDJnVT8rbKH9W1Kx6NhZZJQQJ99CDACYeBjFXJ3w3AAABACOGD59K"
os.environ["AZURE_API_BASE"] = "https://tcoeaiteamgpt41nano.openai.azure.com/openai/deployments/gpt-4.1-nano/chat/completions?api-version=2025-01-01-preview"
os.environ["AZURE_API_VERSION"] = "2024-12-01-preview"
os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"] = "gpt-4.1-nano"

In [6]:
from deepeval.models import AzureOpenAIModel

azure_model = AzureOpenAIModel(
    model="gpt-4.1-nano",                                  # underlying model name
    deployment_name=os.environ["AZURE_OPENAI_LLM_DEPLOYMENT"],
    api_key=os.environ["AZURE_API_KEY"],
    api_version=os.environ["AZURE_API_VERSION"],
    base_url=os.environ["AZURE_API_BASE"],
    temperature=0,
    # If you point this at a GPT-5 / reasoning deployment, remove `temperature`.
)

### Connection check

In [7]:
try:
    print("Judge model responded:", azure_model.generate("Reply with just the word: OK"))
except Exception as e:
    print("Config check failed — review the Azure values above.\n", e)

Judge model responded: ('OK', 2.2e-06)


### Shared helpers & metric definitions

`run_metrics` scores a list of test cases and returns a tidy table. The two metric factories build fresh metric instances for each dataset, so Part 1 and Part 2 can each be run independently (after Setup).

In [8]:
import pandas as pd

# Import DeepEval test case classes.
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

# Import the evaluation metrics used in this notebook.
from deepeval.metrics import (
    SummarizationMetric,
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    GEval,
    BiasMetric,
    ToxicityMetric,
    PIILeakageMetric,
    PromptAlignmentMetric,
)


def run_metrics(test_cases, labels, metrics):
    """Run all metrics on all test cases and return the results as a DataFrame."""

    rows = []

    for tc, label in zip(test_cases, labels):
        for metric in metrics:

            # Get a readable metric name.
            name = (
                getattr(metric, "__name__", None)
                or getattr(metric, "name", None)
                or type(metric).__name__
            )

            try:
                # Execute the metric.
                metric.measure(tc)

                rows.append({
                    "Case": label,
                    "Metric": name,
                    "Score": round(metric.score, 3) if metric.score is not None else None,
                    "Passed": metric.success,
                    "Reason": (metric.reason or "")[:300],
                })

            except Exception as e:
                # Capture metric failures without stopping the evaluation.
                rows.append({
                    "Case": label,
                    "Metric": name,
                    "Score": None,
                    "Passed": None,
                    "Reason": f"error: {e}",
                })

    return pd.DataFrame(rows)


def make_easytravel_metrics():
    """Metrics for transcript summarization."""

    return [
        # Evaluates overall summary quality.
        SummarizationMetric(
            model=azure_model,
            threshold=0.5,
        ),

        # Custom LLM judge comparing generated and reference summaries.
        GEval(
            name="Summary Correctness",
            model=azure_model,
            evaluation_params=[
                LLMTestCaseParams.INPUT,
                LLMTestCaseParams.ACTUAL_OUTPUT,
                LLMTestCaseParams.EXPECTED_OUTPUT,
            ],
            evaluation_steps=[
                "Compare the 'actual output' summary against the 'expected output' reference summary.",
                "Heavily penalize factual contradictions (wrong names, numbers, membership tiers, or resolution status).",
                "Lightly penalize missing key details; ignore differences in wording or style.",
            ],
        ),

        # Checks whether the summary stays relevant/on-topic to the original transcript.
        AnswerRelevancyMetric(
            model=azure_model,
            threshold=0.5,
        ),

        # Flags biased language (gender, racial, political, etc.) in the generated summary.
        BiasMetric(
            model=azure_model,
            threshold=0.5,
        ),

        # Flags toxic or inappropriate language in the generated summary.
        ToxicityMetric(
            model=azure_model,
            threshold=0.5,
        ),

        # Flags unnecessary leakage of personally identifiable information (names, phone numbers,
        # loyalty IDs, etc.) in the generated summary.
        PIILeakageMetric(
            model=azure_model,
            threshold=0.5,
        ),

        # Checks whether the summary follows house-style instructions for EasyTravel summaries.
        PromptAlignmentMetric(
            prompt_instructions=[
                "State the resolution status of the customer's issue (resolved, pending, or escalated).",
                "Do not add opinions, speculation, or details not present in the transcript.",
                "Keep the summary concise (a short paragraph, not a turn-by-turn retelling).",
            ],
            model=azure_model,
            threshold=0.5,
        ),
    ]


def make_policybot_metrics():
    """Metrics for the RAG-based PolicyBot.

    Uses `retrieval_context` = Actual_Context (what the bot actually retrieved) and
    `context` = Expected_Context (the ideal ground-truth passage) on the LLMTestCase,
    so retriever-quality metrics (Contextual Precision/Recall) can compare what was
    retrieved against what should have been retrieved.
    """

    return [
        # Checks if the answer addresses the user's question.
        AnswerRelevancyMetric(
            model=azure_model,
            threshold=0.7,
        ),

        # Checks whether the answer is grounded in what was ACTUALLY retrieved.
        FaithfulnessMetric(
            model=azure_model,
            threshold=0.7,
        ),

        # Checks whether the ACTUALLY retrieved context is relevant to the question.
        ContextualRelevancyMetric(
            model=azure_model,
            threshold=0.7,
        ),

        # Checks whether the most relevant retrieved chunks are ranked/surfaced first.
        ContextualPrecisionMetric(
            model=azure_model,
            threshold=0.7,
        ),

        # Checks whether the retriever pulled back everything needed to answer fully.
        ContextualRecallMetric(
            model=azure_model,
            threshold=0.7,
        ),

        # Custom LLM judge comparing generated and expected answers.
        GEval(
            name="Answer Correctness",
            model=azure_model,
            evaluation_params=[
                LLMTestCaseParams.INPUT,
                LLMTestCaseParams.ACTUAL_OUTPUT,
                LLMTestCaseParams.EXPECTED_OUTPUT,
            ],
            evaluation_steps=[
                "Compare the 'actual output' answer with the 'expected output' ground-truth answer.",
                "Heavily penalize wrong facts or numbers (e.g. a different number of days or dollar amount).",
                "Ignore differences in phrasing when the meaning matches.",
            ],
        ),
    ]


# Number of rows to evaluate (None = evaluate all rows).
N_ROWS = 3


/tmp/ipykernel_987/1666574686.py:4: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


---
# Part 1 — Excel Data Evaluation

Upload **`EasyTravel_Sample_Transcripts_2.xlsx`** and **`PolicyBot_Sample_QA_1.xlsx`** to `/content/` (left sidebar → Files → Upload), then run the sub-sections below.

## 1.1 EasyTravel — Transcript Summarization

**Columns:** `Scenario`, `Input` (transcript), `Expected_Output` (reference summary), `Actual_Output` (AI-generated summary to evaluate).

**Metrics:**
- **SummarizationMetric** – Evaluates the overall quality and completeness of the generated summary.
- **GEval ("Summary Correctness")** – Compares the generated summary against the reference summary using an LLM judge.
- **AnswerRelevancyMetric** – Checks whether the summary stays relevant/on-topic to the original transcript.
- **BiasMetric** – Flags biased language (gender, racial, political, etc.) in the generated summary.
- **ToxicityMetric** – Flags toxic or inappropriate language in the generated summary.
- **PIILeakageMetric** – Flags unnecessary leakage of personally identifiable information in the generated summary.
- **PromptAlignmentMetric** – Checks whether the summary follows house-style instructions (states resolution status, avoids added speculation, stays concise).


In [9]:
# Provide the PATH of the excel and the sheet name as well
df_et = pd.read_excel(
    "/content/EasyTravel_Sample_Transcripts.xlsx",
    sheet_name="Sample Transcripts"
)

# Remove rows missing the transcript or generated summary.
df_et = df_et.dropna(subset=["Input", "Actual_Output"]).reset_index(drop=True)
print(f"Loaded {len(df_et)} transcript rows")
df_et[["Scenario"]].head(10)

Loaded 10 transcript rows


,Scenario
0,Flight cancellation refund — resolved successf...
1,"Seat upgrade — partial resolution, callback pr..."
2,Lost baggage claim — escalated to supervisor
3,"Travel insurance claim query — frustrated, unr..."
4,Loyalty points redemption — resolved with work...
5,Hotel booking amendment — simple resolution
6,"Group booking inquiry — complex, multiple acti..."
7,"Flight rescheduling — medical emergency, empat..."
8,Duplicate charge dispute — escalated to billing
9,Honeymoon package customization — happy custom...


In [18]:
# Select the rows to evaluate (all rows if N_ROWS is None).
rows_et = df_et if N_ROWS is None else df_et.head(N_ROWS)

et_cases, et_labels = [], []

# Convert each dataset row into a DeepEval LLMTestCase.
for _, r in rows_et.iterrows():
    et_cases.append(
        LLMTestCase(
            input=str(r["Input"]),                    # Original transcript
            actual_output=str(r["Actual_Output"]),   # AI-generated summary
            expected_output=str(r["Expected_Output"])# Reference summary
        )
    )

    # Use the scenario name as a readable label in the results.
    et_labels.append(str(r["Scenario"])[:60])

# Run the configured EasyTravel metrics.
et_results = run_metrics(et_cases, et_labels, make_easytravel_metrics())

# Display the complete results table.
print(et_results)

# Save the results to an Excel file.
et_results.to_excel("EasyTravel_Evaluation_Results.xlsx", index=False)

print("Results saved to EasyTravel_Evaluation_Results.xlsx")

Output()

Output()

Output()

Output()

Output()

Output()

                                                Case  \
0  Flight cancellation refund — resolved successf...   
1  Flight cancellation refund — resolved successf...   
2  Seat upgrade — partial resolution, callback pr...   
3  Seat upgrade — partial resolution, callback pr...   
4       Lost baggage claim — escalated to supervisor   
5       Lost baggage claim — escalated to supervisor   

                        Metric  Score  Passed  \
0                Summarization  0.600    True   
1  Summary Correctness [GEval]  0.989    True   
2                Summarization  0.000   False   
3  Summary Correctness [GEval]  0.134   False   
4                Summarization  0.000   False   
5  Summary Correctness [GEval]  1.000    True   

                                              Reason  
0  The score is 0.60 because the summary contains...  
1  The actual output perfectly matches the expect...  
2  The score is 0.00 because the summary contains...  
3  The response incorrectly states the cust

## 1.2 PolicyBot — Policy Q&A (RAG)

**Columns:** `Input` (question), `Expected_Output` (reference answer), `Expected_Context` (ideal ground-truth policy passage), `Actual_Output` (AI-generated answer to evaluate), `Actual_Context` (policy passage the bot actually retrieved).

`Actual_Context` is used as the `retrieval_context` on the `LLMTestCase` (what the bot's retriever actually returned), and `Expected_Context` is used as the `context` (the ideal passage) — this lets retriever-quality metrics compare what was retrieved against what should have been retrieved.

**Metrics:**
- **AnswerRelevancyMetric** – Evaluates whether the generated answer addresses the user's question.
- **FaithfulnessMetric** – Checks whether the answer is grounded in what was actually retrieved.
- **ContextualRelevancyMetric** – Evaluates whether the actually-retrieved context is relevant to the question.
- **ContextualPrecisionMetric** – Checks whether the most relevant retrieved chunks are ranked/surfaced first.
- **ContextualRecallMetric** – Checks whether the retriever pulled back everything needed to answer fully.
- **GEval ("Answer Correctness")** – Compares the generated answer against the reference answer using an LLM judge.


In [20]:
# Load the PolicyBot Q&A dataset from the Excel sheet.
df_pb = pd.read_excel(
    "/content/PolicyBot_Sample_QA.xlsx",
    sheet_name="Sample Questions"
)

# Remove rows with missing inputs or model outputs.
df_pb = df_pb.dropna(subset=["Input", "Actual_Output"]).reset_index(drop=True)

print(df_pb)
# Display the number of valid evaluation samples loaded.
print(f"Loaded {len(df_pb)} Q&A rows")

# Preview the first few questions.
df_pb[["Input"]].head(10)



                                                Input  \
0   How many earned leaves can an employee carry f...   
1   What is the per diem allowance for internation...   
2   How many dependents are covered under medical ...   
3       Who approves expense claims above Rs. 50,000?   
4             Can an employee on probation avail WFH?   
5   What is the maximum number of WFH days per wee...   
6    How is the annual performance rating calculated?   
7   What documents are required for medical reimbu...   
8   What is the policy on software installation on...   
9   How many casual leaves can be clubbed with ear...   
10  What is the sum insured under the base medical...   
11  What is the timeline for reporting a data breach?   
12  What is the gift limit employees may accept fr...   
13  What approvals are required for a Purchase Ord...   
14  How many paternity leave days are employees en...   
15  What is the duration of the Performance Improv...   
16  Is moonlighting allowed at 

,Input
0,How many earned leaves can an employee carry f...
1,What is the per diem allowance for internation...
2,How many dependents are covered under medical ...
3,"Who approves expense claims above Rs. 50,000?"
4,Can an employee on probation avail WFH?
5,What is the maximum number of WFH days per wee...
6,How is the annual performance rating calculated?
7,What documents are required for medical reimbu...
8,What is the policy on software installation on...
9,How many casual leaves can be clubbed with ear...


In [12]:
# Select the rows to evaluate (all rows if N_ROWS is None).
rows_pb = df_pb if N_ROWS is None else df_pb.head(N_ROWS)

pb_cases, pb_labels = [], []

# Convert each dataset row into a DeepEval LLMTestCase.
for _, r in rows_pb.iterrows():
    pb_cases.append(
        LLMTestCase(
            input=str(r["Input"]),                          # User question
            actual_output=str(r["Actual_Output"]),          # AI-generated answer
            expected_output=str(r["Expected_Output"]),      # Reference answer
            retrieval_context=[str(r["Actual_Context"])],   # What the bot ACTUALLY retrieved
            context=[str(r["Expected_Context"])],           # Ideal/ground-truth passage (for Contextual Precision/Recall)
        )
    )

    # Use the input question as a readable label in the results.
    pb_labels.append(str(r["Input"])[:60])

# Run the configured PolicyBot metrics.
pb_results = run_metrics(pb_cases, pb_labels, make_policybot_metrics())

# Display the evaluation results.
print(pb_results)

# Save the results to an Excel file.
pb_results.to_excel("PolicyBot_Evaluation_Results.xlsx", index=False)

print("Results saved to PolicyBot_Evaluation_Results.xlsx")

Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

,Case,Metric,Score,Passed,Reason
0,How many earned leaves can an employee carry f...,Answer Relevancy,0.500,False,The score is 0.50 because the response contain...
1,How many earned leaves can an employee carry f...,Faithfulness,0.000,False,The score is 0.00 because the actual output in...
2,How many earned leaves can an employee carry f...,Contextual Relevancy,1.000,True,The score is 1.00 because the retrieval contex...
3,How many earned leaves can an employee carry f...,Answer Correctness [GEval],0.276,False,The actual output incorrectly states that only...
4,What is the per diem allowance for internation...,Answer Relevancy,1.000,True,The score is 1.00 because all statements in th...
5,What is the per diem allowance for internation...,Faithfulness,1.000,True,The score is 1.00 because there are no contrad...
6,What is the per diem allowance for internation...,Contextual Relevancy,1.000,True,The score is 1.00 because the retrieval contex...
7,What is the per diem allowance for internation...,Answer Correctness [GEval],0.635,True,The actual output correctly states the per die...
8,How many dependents are covered under medical ...,Answer Relevancy,1.000,True,The score is 1.00 because all statements in th...
9,How many dependents are covered under medical ...,Faithfulness,0.000,False,The score is 0.00 because the actual output in...


---
# Part 2 — Multi-turn PolicyBot Dataset

Upload **`policybot_multiturn_output_generation.xlsx`** to `/content/` (left sidebar → Files → Upload),
then run the cells below.

**Columns:** `case_id`, `turn`, `question`, `expected_answer`, `actual_answer`, `deepeval`, `ragas`.

Each `case_id` is one multi-turn conversation. The `deepeval` column only holds a value on the
**last row of each conversation**, containing a ready-made list of `Turn(...)` objects for the
entire conversation (built from the `actual_answer`s). We extract that per conversation and wrap
it in a `ConversationalTestCase`, using `case_id` as the label for the results.


In [ ]:
from deepeval.test_case import Turn, ConversationalTestCase

# Load the multi-turn PolicyBot dataset from the Excel sheet.
df_pb_mt = pd.read_excel(
    "/content/policybot_multiturn_output_generation.xlsx",
    sheet_name="Policy Questions"
)

# The 'deepeval' column only holds a value on the LAST row of each conversation, containing a
# ready-made list of Turn(...) objects for the entire conversation.
conversation_rows = df_pb_mt[df_pb_mt["deepeval"].notna()].reset_index(drop=True)

policybot_conversations, policybot_labels = [], []

for _, row in conversation_rows.iterrows():
    # The 'deepeval' cell is Python source defining a list of Turn(...) objects. Evaluate it in a
    # restricted namespace that only exposes the Turn class (no builtins).
    turns = eval(str(row["deepeval"]), {"Turn": Turn, "__builtins__": {}})

    policybot_conversations.append(ConversationalTestCase(turns=turns))

    # Use the case_id as the label for this conversation in the results.
    policybot_labels.append(str(row["case_id"]))

print(f"Loaded {len(policybot_conversations)} multi-turn PolicyBot conversations: {policybot_labels}")


## Multi-turn metrics

Built in the **same style as Part 1**: a `make_multiturn_policybot_metrics()` factory returns fresh
metric instances, and the shared `run_metrics(...)` helper scores each conversation and returns a table.

This data has no `chatbot_role` and no conversation-level `expected_outcome`, so the following are
**excluded**:
- `RoleAdherenceMetric` — requires `chatbot_role`
- `TurnContextualPrecisionMetric` / `TurnContextualRecallMetric` — require `expected_outcome`

The remaining **9** metrics run on `policybot_conversations`:
- **Core:** TurnRelevancy, KnowledgeRetention, ConversationCompleteness
- **Behavioral:** TopicAdherence *(topics passed to the metric)*
- **Agentic:** GoalAccuracy
- **Custom:** ConversationalGEval, ConversationalDAG
- **Multi-turn RAG:** TurnFaithfulness, TurnContextualRelevancy *(referenceless — no `expected_outcome` needed)*


In [ ]:
from deepeval.metrics import (
    TurnRelevancyMetric, KnowledgeRetentionMetric, ConversationCompletenessMetric,
    TopicAdherenceMetric, GoalAccuracyMetric,
    TurnFaithfulnessMetric, TurnContextualRelevancyMetric,
    ConversationalGEval, ConversationalDAGMetric,
)
from deepeval.test_case import MultiTurnParams
from deepeval.metrics.dag import DeepAcyclicGraph
from deepeval.metrics.conversational_dag import (
    ConversationalTaskNode, ConversationalBinaryJudgementNode, ConversationalVerdictNode,
)

# Topics the PolicyBot assistant is allowed to discuss (for TopicAdherenceMetric).
POLICYBOT_TOPICS = [
    "leave policy (earned/sick/casual/maternity/paternity leave)",
    "leave application and approval process",
    "attendance and unauthorized absence",
    "HR record-keeping for leave",
]

def _build_conversational_dag():
    """A tiny decision tree for ConversationalDAGMetric: did the assistant resolve every request?"""
    resolved = ConversationalBinaryJudgementNode(
        criteria="Did the assistant fully resolve every request the user raised?",
        children=[
            ConversationalVerdictNode(verdict=False, score=0),
            ConversationalVerdictNode(verdict=True, score=10),
        ],
    )
    root = ConversationalTaskNode(
        instructions="Review the conversation and summarize whether each user request was resolved.",
        output_label="Summary",
        evaluation_params=[MultiTurnParams.ROLE, MultiTurnParams.CONTENT],
        children=[resolved],
    )
    return DeepAcyclicGraph(root_nodes=[root])

def make_multiturn_policybot_metrics():
    """Multi-turn metrics for PolicyBot conversations.

    Excludes RoleAdherenceMetric (needs `chatbot_role`) and TurnContextualPrecisionMetric /
    TurnContextualRecallMetric (need `expected_outcome`), since this data doesn't carry either field.
    """
    return [
        # --- Core conversation metrics ---
        TurnRelevancyMetric(model=azure_model, threshold=0.5),
        KnowledgeRetentionMetric(model=azure_model, threshold=0.5),
        ConversationCompletenessMetric(model=azure_model, threshold=0.5),
        # --- Behavioral compliance ---
        TopicAdherenceMetric(relevant_topics=POLICYBOT_TOPICS, model=azure_model, threshold=0.5),
        # --- Agentic ---
        GoalAccuracyMetric(model=azure_model, threshold=0.5),
        # --- Custom ---
        ConversationalGEval(
            name="Professionalism",
            criteria="Determine whether the assistant stayed professional, accurate, and "
                     "helpful toward the employee across the whole conversation.",
            evaluation_params=[MultiTurnParams.ROLE, MultiTurnParams.CONTENT],
            model=azure_model, threshold=0.5,
        ),
        ConversationalDAGMetric(name="Requests Resolved", dag=_build_conversational_dag(),
                                model=azure_model),
        # --- Multi-turn RAG (referenceless) ---
        TurnFaithfulnessMetric(model=azure_model, threshold=0.5),
        TurnContextualRelevancyMetric(model=azure_model, threshold=0.5),
    ]


In [ ]:
# Score each multi-turn PolicyBot conversation (same run_metrics helper as Part 1).
df_mt = run_metrics(policybot_conversations, policybot_labels, make_multiturn_policybot_metrics())

# Use case_id as the label column in the output.
df_mt = df_mt.rename(columns={"Case": "case_id"})
df_mt


In [ ]:
# Save the multi-turn results to Excel (case_id first, then each metric's output).
df_mt.to_excel("PolicyBot_Multiturn_Evaluation_Results.xlsx", index=False)
print("Results saved to PolicyBot_Multiturn_Evaluation_Results.xlsx")
